In [16]:
import tiktoken
from openai import OpenAI
import os
import ast

import pandas as pd
import numpy as np

# 嵌入模型
openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key,
)
embedding_model = 'Qwen/Qwen3-Embedding-8B'


In [4]:
df = pd.read_csv('output/embedding_1000.csv')

df['embedding_vec'] = df['embedding'].apply(ast.literal_eval)
df.head(1)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,count_token,embedding,embedding_vec
0,0,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title:0 where does one start...and stop....,264,"[-0.015835756435990334, -0.013079162687063217,...","[-0.015835756435990334, -0.013079162687063217,..."


In [17]:
# 在向量空间中，语义相似的词或文本单位。距离靠近

def cosine_distance(vec1, vec2):
    """
    计算两个向量之间的余弦距离
    :param vec1: 向量1
    :param vec2: 向量2
    :return: 余弦距离
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def embedding_text(text, model):
    """
    通过Embedding模型处理文本数据
    :param text: 需要处理的文本
    :param model: 嵌入模型
    :return: 文本向量
    """
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding


def search_by_keyword(df, keyword, top_n=10, print_flag=True):
    """
    根据关键词搜索最相似的文本
    :param print_flag: 是否打印结果
    :param df: 数据框
    :param keyword: 关键词
    :param top_n: 返回最相似的n个文本
    :return: 最相似的n个文本
    """
    word_embedding = embedding_text(keyword, model=embedding_model)
    # 计算相识度
    df['similarity'] = df['embedding_vec'].apply(lambda x: cosine_distance(x, word_embedding))
    result = (
        df.sort_values(by='similarity', ascending=False)
        .head(top_n)['combined'].str.replace('Title:', '')
        .str.replace('; Content:', ';')
    )
    if print_flag:
        for i in result:
            print(i)

    return result

In [20]:
search_by_keyword(df, 'Suck', top_n=3)

0      where does one  start...and stop... with a tre...
1                                      Arrived in pieces
2              It isn't blanc mange, but isn't bad . . .
3            These also have SALT and it's not sea salt.
4                                 Happy with the product
                             ...                        
995                                           Delicious!
996                                  Good Training Treat
997                               Jamica Me Crazy Coffee
998                                        Party Peanuts
999                                  I love Maui Coffee!
Name: Summary, Length: 1000, dtype: str;0      Wanted to save some to bring to my Chicago fam...
1      Not pleased at all. When I opened the box, mos...
2      I'm not sure that custard is really custard wi...
3      I like the fact that you can see what you're g...
4      My dog was suffering with itchy skin.  He had ...
                             ...                

1    0      where does one  start...and stop... wit...
6    0      where does one  start...and stop... wit...
5    0      where does one  start...and stop... wit...
Name: combined, dtype: str

In [21]:
df

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,count_token,embedding,embedding_vec,similarity
0,0,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title:0 where does one start...and stop....,264,"[-0.015835756435990334, -0.013079162687063217,...","[-0.015835756435990334, -0.013079162687063217,...",0.469629
1,1,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title:0 where does one start...and stop....,264,"[-0.015358959324657917, -0.012955457903444767,...","[-0.015358959324657917, -0.012955457903444767,...",0.474035
2,2,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,Title:0 where does one start...and stop....,264,"[-0.015835756435990334, -0.013079162687063217,...","[-0.015835756435990334, -0.013079162687063217,...",0.469629
3,3,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title:0 where does one start...and stop....,264,"[-0.015592706389725208, -0.013130700215697289,...","[-0.015592706389725208, -0.013130700215697289,...",0.471630
4,4,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...,Title:0 where does one start...and stop....,264,"[-0.015493586659431458, -0.01285263430327177, ...","[-0.015493586659431458, -0.01285263430327177, ...",0.471177
5,5,B008PSM0BQ,A3OUFIMGL2K6RS,4,Good Sauce,This is a good all purpose sauce. Has good fl...,Title:0 where does one start...and stop....,264,"[-0.01547937747091055, -0.013075383380055428, ...","[-0.01547937747091055, -0.013075383380055428, ...",0.471651
6,6,B008YA1LQK,A9YEAAQVHFUTX,5,Blackcat,Great coffee! Love all Green Mountain coffee ...,Title:0 where does one start...and stop....,264,"[-0.015358959324657917, -0.012955457903444767,...","[-0.015358959324657917, -0.012955457903444767,...",0.474035
7,7,B001KP6B98,ABWCUS3HBDZRS,5,Excellent product,After scouring every store in town for orange ...,Title:0 where does one start...and stop....,264,"[-0.015592706389725208, -0.013130700215697289,...","[-0.015592706389725208, -0.013130700215697289,...",0.471630
8,8,B008YA1LQK,A2RSB6FVQ9K9OD,5,Bulk k-Cups,This is the best way to buy coffee for my offi...,Title:0 where does one start...and stop....,264,"[-0.015592706389725208, -0.013130700215697289,...","[-0.015592706389725208, -0.013130700215697289,...",0.471630
9,9,B001E5E2QI,A23WYVBCNE75X1,3,It's Okay,"Next time, I will buy Gevalia Irish Cream deca...",Title:0 where does one start...and stop....,264,"[-0.015621102415025234, -0.013095886446535587,...","[-0.015621102415025234, -0.013095886446535587,...",0.471368
